<a href="https://colab.research.google.com/github/tien10022001/txldl/blob/main/lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Bài 1: Khám phá dữ liệu

import pandas as pd

# Đọc file với đúng định dạng (dấu ; phân cách và dấu , là thập phân)
df = pd.read_csv('ITA105_Lab_1.csv', sep=';', decimal=',')

# 1. Kiểm tra kích thước dữ liệu
print(f"Kích thước dữ liệu: {df.shape[0]} dòng và {df.shape[1]} cột")

# 2. Xem thống kê mô tả của các cột số
print("\nThống kê mô tả các cột số:")
display(df.describe())

# 3. Kiểm tra giá trị thiếu trong các cột
print("\nSố lượng giá trị thiếu trong mỗi cột:")
print(df.isnull().sum())

#df.shape cho biết số lượng hàng/cột.
#df.describe() cho thấy các chỉ số như trung bình, lớn nhất, nhỏ nhất để phát hiện các con số "lạ".

Kích thước dữ liệu: (200, 6)

Thống kê mô tả:


,ProductID,Price,Rating
count,200.000000,1.950000e+02,200.000000
mean,100.500000,1.430049e+16,2.945000
std,57.879185,1.840537e+16,2.050119
min,1.000000,5.087313e+12,0.000000
25%,50.750000,3.959229e+15,1.000000
50%,100.500000,5.536681e+15,3.000000
75%,150.250000,8.583426e+15,5.000000
max,200.000000,6.376793e+16,6.000000



Giá trị thiếu trong các cột:
ProductID        0
Category         5
Price            5
StockQuantity    5
Rating           0
Description      0
dtype: int64


In [2]:
# Bài 2: Xử lý dữ liệu thiếu
# 1. Phát hiện giá trị thiếu
print("Giá trị thiếu ban đầu:\n", df.isnull().sum())

# 2. Điền giá trị thiếu (Imputation)
# Price: điền bằng giá trị trung bình (mean)
df['Price'] = df['Price'].fillna(df['Price'].mean())

# StockQuantity: Chuyển về số rồi điền bằng trung vị (median)
df['StockQuantity'] = pd.to_numeric(df['StockQuantity'], errors='coerce')
df['StockQuantity'] = df['StockQuantity'].fillna(df['StockQuantity'].median())

# Category và Description: điền bằng giá trị xuất hiện nhiều nhất (mode) hoặc mặc định
df['Category'] = df['Category'].fillna(df['Category'].mode()[0])
df['Description'] = df['Description'].fillna("No description available")

# 3. So sánh với phương pháp dropna()
df_dropped = pd.read_csv('ITA105_Lab_1.csv', sep=';').dropna()
print(f"\nSố dòng sau khi điền thiếu (fillna): {len(df)}")
print(f"Số dòng nếu dùng phương pháp xóa (dropna): {len(df_dropped)}")

Số dòng sau khi điền thiếu: 200
Số dòng nếu dùng dropna(): 186


In [ ]:
# Bài 3: Xử lý dữ liệu lỗi
# 1. Xử lý Price: Các giá trị bị mất dấu thập phân nên rất lớn, ta chia cho 10^14
df.loc[df['Price'] > 1000, 'Price'] = df['Price'] / 10**14

# 2. Xử lý StockQuantity: Giá trị âm biến thành 0
df.loc[df['StockQuantity'] < 0, 'StockQuantity'] = 0

# 3. Lọc Rating: Chỉnh các giá trị ngoài khoảng [1, 5]
df.loc[df['Rating'] > 5, 'Rating'] = 5
df.loc[df['Rating'] < 1, 'Rating'] = 1

print("Dữ liệu sau khi xử lý lỗi (5 dòng đầu):")
display(df.head())

In [ ]:
# Bài 4: Làm mượt dữ liệu nhiễu
import matplotlib.pyplot as plt

# --- 1. Áp dụng Moving Average (Trung bình trượt) ---
# Chúng ta chọn cửa sổ (window) là 5, nghĩa là mỗi điểm mới sẽ là trung bình của 5 điểm trước đó.
# min_periods=1 giúp tính toán ngay cả ở những dòng đầu tiên khi chưa đủ 5 điểm.
df['Price_MA'] = df['Price'].rolling(window=5, min_periods=1).mean()

# --- 2. Vẽ biểu đồ line so sánh trước và sau khi làm mượt ---
plt.figure(figsize=(14, 7)) # Thiết kế kích thước biểu đồ rộng rãi

# Vẽ đường giá gốc (Màu xanh, nét mờ alpha=0.4 để dễ nhìn đường làm mượt)
plt.plot(df['ProductID'].head(60), df['Price'].head(60),
         label='Giá gốc (Original Price)',
         color='blue', linestyle='--', alpha=0.4, marker='o')

# Vẽ đường giá đã làm mượt (Màu đỏ, nét đậm hơn)
plt.plot(df['ProductID'].head(60), df['Price_MA'].head(60),
         label='Giá làm mượt (Moving Average - 5)',
         color='red', linewidth=2.5)

# Thêm các thành phần bổ trợ cho biểu đồ
plt.title('SO SÁNH DỮ LIỆU GIÁ TRƯỚC VÀ SAU KHI LÀM MƯỢT (60 sản phẩm đầu)', fontsize=14)
plt.xlabel('Mã sản phẩm (ProductID)', fontsize=12)
plt.ylabel('Giá (USD)', fontsize=12)
plt.legend() # Hiển thị ghi chú cho các đường
plt.grid(True, linestyle=':', alpha=0.6) # Thêm lưới cho biểu đồ dễ quan sát

# Hiển thị biểu đồ
plt.show()

In [3]:
# Bài 5: Chuẩn hóa dữ liệu
# 1. Category thành chữ thường
df['Category'] = df['Category'].str.lower()

# 2. Loại bỏ ký tự thừa trong Description (?, !, ...)
df['Description'] = df['Description'].str.replace(r'[^\w\s]', '', regex=True).str.strip()

# 3. Chuyển USD sang VND (Giả sử 1 USD = 25,000 VND)
df['Price_VND'] = df['Price'] * 25000

print("Dữ liệu sau khi chuẩn hóa:")
display(df[['Category', 'Description', 'Price_VND']].head())

Dữ liệu sau khi chuẩn hóa:


,Category,Description,Price_VND
0,clothing,Not worth it,1.282643e+20
1,home,Good product,1.137872e+20
2,electronics,Cheap and useful,1.284410e+20
3,clothing,Cheap and useful,5.046617e+20
4,clothing,Cheap and useful,1.167623e+20


In [4]:
# Tính tổng giá trị hàng tồn kho (Price * StockQuantity) cho mỗi sản phẩm
df['Inventory_Value'] = df['Price'] * df['StockQuantity']

# Tính trung bình giá theo từng Category
avg_price_by_cat = df.groupby('Category')['Price'].mean()
print("Giá trung bình theo danh mục:")
print(avg_price_by_cat)

# Xuất file kết quả cuối cùng
df.to_csv('Lab1_Hoan_Thanh.csv', index=False)

Giá trung bình theo danh mục:
Category
books          1.583735e+16
clothing       1.379462e+16
electronics    1.497365e+16
home           1.299150e+16
Name: Price, dtype: float64
